# Adult (DP-GAN) 

Author: Ilse Harmers \
Last modified: April 21, 2026

In [ ]:
# Importing libraries.
import numpy as np
import pandas as pd
from snsynth import Synthesizer
import snsynth.transform as tf
import utils
import warnings
warnings.filterwarnings("ignore", message = r"Using", category = FutureWarning)

In [ ]:
# Importing train set.
adult_train = pd.read_csv("./train-test-datasets/adult/adult_train.csv")
print(adult_train.columns.to_list())
adult_train.describe()

In [ ]:
# Setting up preprocessor table transformer.

tt = tf.TableTransformer([
    tf.MinMaxTransformer(lower = adult_train["age"].min(), upper = adult_train["age"].max(), 
                         negative = False), # age; scaling to range (0, 1)
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # workclass
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # education-num
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # marital-status
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # occupation
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # relationship
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # race
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # sex
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = 0, upper = np.log(adult_train["capital-gain"].max() + 1),
                             negative = False) # capital-gain; scaling to range (0, 1)
    ]),
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = 0, upper = np.log(4400.0 + 1),   # 1 id with 4356.0; rounded up w.r.t. privacy
                             negative = False) # capital-loss; scaling to range (0, 1)
    ]),
    tf.MinMaxTransformer(lower = adult_train["hours-per-week"].min(), 
                         upper = adult_train["hours-per-week"].max(), 
                         negative = False), # hours-per-week; scaling to range (0, 1)
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # native-country
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # income
])

In [ ]:
# Defining delta as the inverse of the size of the dataset: if a dataset has 4 * 10^4 rows, then delta = 10^(-5).
delta = 10**np.floor(np.log10(1/adult_train.shape[0]))
print(delta)

# Defining epsilon value.
epsi = 2

# Defining differentially private GAN model: [dpgan].
model = "dpgan"

In [ ]:
r = 1
while r < 6:
    
    print(f"Run {r}")
    
    # Synthesizing the dataset with DP-GAN. 
    synth = Synthesizer.create(model, epsilon = epsi, delta = delta, batch_size = 512, epochs = 10000, epoch_time = True)
    synth.fit(adult_train, transformer = tt, preprocessor_eps = 0.0)
    
    r += 1

In [ ]:
# Saving the results on runtime.
time_per_run = pd.DataFrame({"time": [0.755240, 0.750519, 0.747756, 0.747872, 0.748782]}, index = [f"run{i}" for i in range(1, 6)])
time_per_run.to_csv(f"./Runtimes/adult_runtime_{model}.csv")
time_per_run

In [ ]:
# Computing average runtime across the runs.
print("Mean runtime [s/epoch]:", np.mean(time_per_run["time"]))
print("Standard deviation runtime [s/epoch]:", np.std(time_per_run["time"]))